<a href="https://colab.research.google.com/github/kridtapon/HMA-Overdrive-/blob/main/HMA_Overdrive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install vectorbt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.6/527.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.7 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import yfinance as yf
import vectorbt as vbt

# Function to calculate Weighted Moving Average (WMA)
def wma(series, period):
    weights = np.arange(1, period + 1)
    return series.rolling(period).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

# Function to calculate Hull Moving Average (HMA)
def hma(series, period):
    half_length = int(period / 2)
    sqrt_length = int(np.sqrt(period))

    wma_half = wma(series, half_length)
    wma_full = wma(series, period)
    hma_series = wma(2 * wma_half - wma_full, sqrt_length)

    return hma_series

# Function to calculate ATR
def calculate_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = tr.rolling(window=period).mean()
    return atr

# Define the stock symbol and time period
symbol = 'META'  # Example: META (Facebook)
start_date = '2019-01-01'
end_date = '2025-01-01'

# Download the data
df = yf.download(symbol, start=start_date, end=end_date)
df.columns = ['Close', 'High', 'Low', 'Open', 'Volume']

# Calculate Hull Moving Averages
df['HMA_Short'] = hma(df['Close'], 10)  # Short-term HMA
df['HMA_Long'] = hma(df['Close'], 50)   # Long-term HMA

# Calculate ATR
df['ATR'] = calculate_atr(df)

# Define ATR Overextension multiplier (e.g., 2x ATR for overextension)
atr_multiplier = 2

# Generate Entry and Exit signals based on HMA crossover & ATR Overextension
df['Entry'] = (df['HMA_Short'] > df['HMA_Long']) & (df['Close'] > df['Close'].shift(10) + atr_multiplier * df['ATR'])
df['Exit'] = (df['HMA_Short'] < df['HMA_Long']) & (df['Close'] < df['Close'].shift(10) - atr_multiplier * df['ATR'])

# Filter data for the test period (2020-2025)
df = df[(df.index.year >= 2020) & (df.index.year <= 2025)]

# Backtest using vectorbt
portfolio = vbt.Portfolio.from_signals(
    close=df['Close'],
    entries=df['Entry'],
    exits=df['Exit'],
    init_cash=100_000,
    fees=0.001
)

# Display performance metrics
print(portfolio.stats())

# Plot equity curve
portfolio.plot().show()

[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.11/dist-packages/vectorbt/generic/stats_builder.py:396: UserWarning: Metric 'sharpe_ratio' requires frequency to be set
  warnings.warn(warning_message)
/usr/local/lib/python3.11/dist-packages/vectorbt/generic/stats_builder.py:396: UserWarning: Metric 'calmar_ratio' requires frequency to be set
  warnings.warn(warning_message)
/usr/local/lib/python3.11/dist-packages/vectorbt/generic/stats_builder.py:396: UserWarning: Metric 'omega_ratio' requires frequency to be set
  warnings.warn(warning_message)
/usr/local/lib/python3.11/dist-packages/vectorbt/generic/stats_builder.py:396: UserWarning: Metric 'sortino_ratio' requires frequency to be set
  warnings.warn(warning_message)


Start                         2020-01-02 00:00:00
End                           2024-12-31 00:00:00
Period                                       1258
Start Value                              100000.0
End Value                           310999.445489
Total Return [%]                       210.999445
Benchmark Return [%]                   180.172856
Max Gross Exposure [%]                      100.0
Total Fees Paid                       7819.396005
Max Drawdown [%]                        36.760048
Max Drawdown Duration                       612.0
Total Trades                                   22
Total Closed Trades                            22
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]                            40.909091
Best Trade [%]                          53.573359
Worst Trade [%]                         -9.890704
Avg Winning Trade [%]                   23.231699
Avg Losing Trade [%]                    -4.377669
